# 🛒 PCA y Feature Selection en E-Commerce
## Retail Rocket Dataset - Reducción Dimensional para Predicción de Compras

**Autora:** Milagros Cancela  
**Fecha:** Diciembre 2024  
**Dataset:** [Retail Rocket - Kaggle](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset)

## 1. Configuración e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Librerías importadas correctamente')

## 2. Carga de Datos

In [ ]:
try:
    import kagglehub
    path = kagglehub.dataset_download('retailrocket/ecommerce-dataset')
    events = pd.read_csv(f'{path}/events.csv')
except:
    events = pd.read_csv('events.csv')

print(f'Dataset cargado: {events.shape[0]:,} eventos')
events.head()

In [ ]:
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date'] = events['timestamp'].dt.date
print(f'Rango: {events["date"].min()} a {events["date"].max()}')
print(events['event'].value_counts())

## 3. Feature Engineering

In [ ]:
max_date = events['timestamp'].max()

event_pivot = events.pivot_table(
    index='visitorid', columns='event', values='timestamp',
    aggfunc='count', fill_value=0
).reset_index()
event_pivot.columns = ['visitorid', 'total_addtocart', 'total_transactions', 'total_views']

user_activity = events.groupby('visitorid').agg({
    'timestamp': ['count', 'min', 'max'],
    'itemid': 'nunique',
    'transactionid': lambda x: x.notna().sum()
}).reset_index()
user_activity.columns = ['visitorid', 'total_events', 'first_event', 'last_event', 'unique_items', 'transaction_count']
user_activity['days_active'] = (user_activity['last_event'] - user_activity['first_event']).dt.days + 1
user_activity['recency'] = (max_date - user_activity['last_event']).dt.days
user_activity['avg_events_per_day'] = user_activity['total_events'] / user_activity['days_active'].clip(lower=1)

In [ ]:
user_features = event_pivot.merge(user_activity, on='visitorid', how='left')
user_features['view_to_cart_ratio'] = user_features['total_addtocart'] / user_features['total_views'].clip(lower=1)
user_features['cart_to_purchase_ratio'] = user_features['total_transactions'] / user_features['total_addtocart'].clip(lower=1)
user_features['conversion_rate'] = user_features['total_transactions'] / user_features['total_events'].clip(lower=1)
user_features['cart_abandonment_rate'] = np.where(user_features['total_addtocart'] > 0, 1 - (user_features['total_transactions'] / user_features['total_addtocart']), 0)
user_features['items_per_session'] = user_features['unique_items'] / user_features['days_active'].clip(lower=1)
user_features['is_buyer'] = (user_features['total_transactions'] > 0).astype(int)

user_features_clean = user_features.drop(columns=['visitorid', 'first_event', 'last_event'])
user_features_clean = user_features_clean.replace([np.inf, -np.inf], 0).fillna(0)
print(f'Dataset: {user_features_clean.shape}')
print(user_features_clean['is_buyer'].value_counts(normalize=True))

## 4. Preparación para Modelado

In [ ]:
X = user_features_clean.drop(columns=['is_buyer'])
y = user_features_clean['is_buyer']
feature_names = X.columns.tolist()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

## 5. PCA

In [ ]:
pca_full = PCA()
pca_full.fit(X_train)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained_variance)+1), explained_variance, color='#7B4B94')
axes[0].set_title('Varianza Individual')
axes[1].plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'o-', color='#7B4B94')
axes[1].axhline(y=0.90, color='red', linestyle='--')
axes[1].set_title('Varianza Acumulada')
plt.tight_layout()
plt.show()

print(f'Componentes para 90%: {np.argmax(cumulative_variance >= 0.90) + 1}')

## 6. Feature Selection - Filter Methods

In [ ]:
f_scores, p_values = f_classif(X_train, y_train)
f_test_df = pd.DataFrame({'Feature': feature_names, 'F-Score': f_scores}).sort_values('F-Score', ascending=False)
print('Top 10 F-Test:')
print(f_test_df.head(10))

mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({'Feature': feature_names, 'MI Score': mi_scores}).sort_values('MI Score', ascending=False)
print('\nTop 10 Mutual Information:')
print(mi_df.head(10))

## 7. Feature Selection - RFE

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=10, step=1)
rfe.fit(X_train, y_train)

rfe_selected = [feat for feat, selected in zip(feature_names, rfe.support_) if selected]
print('Features RFE:')
for i, f in enumerate(rfe_selected, 1): print(f'{i}. {f}')

## 8. Comparación de Métodos

In [ ]:
def evaluate(X_tr, X_te, y_tr, y_te, name):
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    y_proba = rf.predict_proba(X_te)[:, 1]
    y_pred = rf.predict(X_te)
    return {'Method': name, 'N_Features': X_tr.shape[1], 'ROC-AUC': roc_auc_score(y_te, y_proba), 'F1': f1_score(y_te, y_pred)}

results = []
results.append(evaluate(X_train, X_test, y_train, y_test, 'Baseline'))

pca_5 = PCA(n_components=5)
X_train_pca = pca_5.fit_transform(X_train)
X_test_pca = pca_5.transform(X_test)
results.append(evaluate(X_train_pca, X_test_pca, y_train, y_test, 'PCA (5)'))

f_mask = [f in f_test_df.head(10)['Feature'].tolist() for f in feature_names]
results.append(evaluate(X_train[:, f_mask], X_test[:, f_mask], y_train, y_test, 'F-Test Top10'))

mi_mask = [f in mi_df.head(10)['Feature'].tolist() for f in feature_names]
results.append(evaluate(X_train[:, mi_mask], X_test[:, mi_mask], y_train, y_test, 'MI Top10'))

results.append(evaluate(X_train[:, rfe.support_], X_test[:, rfe.support_], y_train, y_test, 'RFE Top10'))

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(results_df)

## 9. Conclusiones

In [ ]:
best = results_df.iloc[0]
print(f'Mejor método: {best["Method"]}')
print(f'ROC-AUC: {best["ROC-AUC"]:.4f}')
print(f'Features: {best["N_Features"]}')
print('\nTop 5 features para predicción de compra:')
for i, f in enumerate(rfe_selected[:5], 1): print(f'{i}. {f}')